# Computer Vision (CV) - Homework 4
## Natural Image Classification on Intel Dataset using CNNs
**Course Instructor:** Dr. Seifipour  
**Semester:** Spring 1404-1405 (Second Semester)  
**Group Members:**  
* Student 1: [Placeholder: Name] - [Placeholder: Student ID]  
* Student 2: [Placeholder: Name] - [Placeholder: Student ID]  
---
### Overview
In this homework, we design, train, evaluate, and improve a multi-class Convolutional Neural Network (CNN) in **TensorFlow/Keras** to classify natural images from the **Intel Image Classification** dataset. The dataset includes 6 categories: buildings, forest, glacier, mountain, sea, and street.


## Setup & Dataset Loading
We download the Intel Image Classification dataset directly from Kaggle into the local Colab SSD storage `/content/` for high-speed training reads.

### Option 1 (Recommended): Download Directly from Kaggle
**Fill in your Kaggle credentials in the cell below and run it.** To find them:
1. Go to **kaggle.com** → your profile picture → **Settings** → scroll to **API**.
2. Click **Create New API Token**. You will see (or download) your `username` and `key`.
3. Paste both values into the variables at the top of the code cell below.

### Option 2 (Alternative): Extract from Google Drive
If you already have `train.zip`/`.rar` and `test.zip`/`.rar` in your Google Drive, you can mount your Drive and extract them locally.


### Option 1: Kaggle API Setup and Download


In [ ]:
# ============================================================
# FILL IN YOUR KAGGLE CREDENTIALS HERE
# (Find them at kaggle.com -> Settings -> API -> Create New Token)
# ============================================================
KAGGLE_USERNAME = 'your_kaggle_username'  # <-- Replace this
KAGGLE_KEY      = 'your_kaggle_api_key'   # <-- Replace this

import os
import json

# Write kaggle.json from the credentials pasted above
os.makedirs('/root/.kaggle', exist_ok=True)
kaggle_creds = {'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_creds, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured.')

# Download and unzip the dataset directly into /content
!pip install -q kaggle
print('Downloading intel-image-classification from Kaggle...')
!kaggle datasets download -d puneet6060/intel-image-classification --unzip -p /content
print('Dataset downloaded and unzipped successfully.')


### Option 2 (Alternative Backup): Mount Google Drive & Extract Archives


In [ ]:
# Run this cell ONLY if you are NOT using Option 1 (Kaggle)
# from google.colab import drive
# import zipfile
# import os
# import rarfile  # Install first: !apt-get install unrar && pip install rarfile

# drive.mount('/content/drive')

# # Paths on Google Drive
# drive_train_path = '/content/drive/MyDrive/Data/train.zip'
# drive_test_path = '/content/drive/MyDrive/Data/test.zip'

# # Local folders
# local_train_dir = '/content/seg_train'
# local_test_dir = '/content/seg_test'

# def extract_archive(archive_path, extract_to):
#     if not os.path.exists(archive_path):
#         print(f'{archive_path} not found.')
#         return
#     os.makedirs(extract_to, exist_ok=True)
#     print(f'Extracting {archive_path}...')
#     if archive_path.endswith('.zip'):
#         with zipfile.ZipFile(archive_path, 'r') as zip_ref:
#             zip_ref.extractall(extract_to)
#     elif archive_path.endswith('.rar'):
#         with rarfile.RarFile(archive_path) as rf:
#             rf.extractall(extract_to)
#     print('Done.')

# # extract_archive(drive_train_path, local_train_dir)
# # extract_archive(drive_test_path, local_test_dir)


### Root Directory Auto-Discovery Helper
Depending on how the dataset was extracted (either via Kaggle or Google Drive), the class folders (e.g. `buildings`, `forest`) might be nested inside another directory. The helper function `find_dataset_root` dynamically discovers the correct directory path containing the 6 classes, ensuring that dataset loading runs smoothly.


In [ ]:
import os

def find_dataset_root(base_path, expected_subdirs):
    for root, dirs, files in os.walk(base_path):
        if all(sub in dirs for sub in expected_subdirs):
            return root
    return base_path


## Section 1: Data Exploration & Loading
In this section, we load the images and set up the validation split:
* Total Training Dataset size: 14,034 images.
* We separate **3,000 images** for **validation** and use the remaining **11,034 images** for **training**.
* To split exactly 3,000 images for validation, we compute the split percentage: $3000 / 14034 \approx 0.21377$. We use `validation_split=0.21377` with `seed=42` to ensure consistent data splitting.
* All images are loaded with spatial dimensions of $150 \times 150$ and three RGB channels.


In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

# Verify GPU is available
gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f'GPU is available: {gpu_devices[0]}')
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print('GPU is NOT available. Running on CPU.')

classes = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

# Search both possible paths (from Kaggle extraction or Drive extraction)
train_root = find_dataset_root('/content/seg_train', classes)
test_root = find_dataset_root('/content/seg_test', classes)

print(f'Discovered Train Root: {train_root}')
print(f'Discovered Test Root: {test_root}')

IMG_SIZE = (150, 150)
BATCH_SIZE = 64

# Count total files to compute validation ratio
total_train_images = sum([len(files) for r, d, files in os.walk(train_root)])
print(f'Total train images found: {total_train_images}')
val_split = 3000 / total_train_images
print(f'Computed Validation Split Ratio: {val_split:.6f} (~{val_split * 100:.3f}%)')

# Load training subset
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=val_split,
    subset='training',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# Load validation subset
val_ds = tf.keras.utils.image_dataset_from_directory(
    train_root,
    validation_split=val_split,
    subset='validation',
    seed=42,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# Load test set (shuffle=False for easier performance report analysis)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_root,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False
)

class_names = train_ds.class_names
print('Classes loaded:', class_names)


In [ ]:
# Retrieve and print the shape of a single batch
for images, labels in train_ds.take(1):
    print(f'Image batch shape: {images.shape} (Batch size, Height, Width, Channels)')
    print(f'Label batch shape: {labels.shape} (Batch size,)')
    print(f'Number of training batches: {len(train_ds)}')
    print(f'Number of validation batches: {len(val_ds)}')


### Display Sample Images
Below, we display a grid of sample training images along with their ground truth class labels.


In [ ]:
plt.figure(figsize=(12, 8))
for images, labels in train_ds.take(1):
    for i in range(6):
        ax = plt.subplot(2, 3, i + 1)
        plt.imshow(images[i].numpy().astype('uint8'))
        plt.title(class_names[labels[i]])
        plt.axis('off')
plt.suptitle('Sample Training Images', fontsize=16)
plt.tight_layout()
plt.show()


### Discussion on Image Shape & Flattening Requirements

#### 1. Dimensions of the Image
Each loaded image is a 3D tensor with shape `(150, 150, 3)`:
* **Width**: 150 pixels.
* **Height**: 150 pixels.
* **Channels**: 3 color channels (Red, Green, Blue).
Thus, a raw image has $150 \times 150 \times 3 = 67,500$ pixel values.

#### 2. Why must the feature maps be Flattened?
Fully connected (Dense) layers require their input to be structured as a 1D vector (or matrix of shape `[batch_size, num_features]`).

* **Spatial vs. Flat Representation**: Convolutional and Pooling layers compute spatial representations (height, width, feature channels). In contrast, Dense layers learn global relationships between elements and perform matrix multiplications of the form $Y = W \cdot X + B$. The input vector $X$ must be a flat 1D array.
* **Role of Flattening**: The `Flatten` layer reshapes a multi-dimensional feature map (e.g. `(batch_size, 9, 9, 128)`) into a 1D feature representation (e.g. `(batch_size, 10368)`) before transferring it to the Dense hidden layer. It does not alter the actual numbers, only their spatial alignment.


## Section 2: Preprocessing & Data Augmentation
Data augmentation artificially introduces diversity into the training dataset. We define:
1. **Normalization**: Scaling the pixels from range $[0, 255]$ to $[0, 1]$ using a Rescaling layer.
2. **Augmentation**: Applying random horizontal flips and random rotations to training images.


In [ ]:
# Sequential layer grouping the data augmentations
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal', name='random_flip'),
    tf.keras.layers.RandomRotation(0.15, name='random_rotation'),
    tf.keras.layers.RandomZoom(0.1, name='random_zoom')
], name='data_augmentation')

# Preview the effect of data augmentation on a single training image
plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    first_image = images[0]
    for i in range(6):
        ax = plt.subplot(3, 3, i + 1)
        augmented_image = data_augmentation(tf.expand_dims(first_image, 0))
        plt.imshow(augmented_image[0].numpy().astype('uint8'))
        plt.axis('off')
plt.suptitle('Data Augmentation Previews on a Sample Image', fontsize=14)
plt.tight_layout()
plt.show()


### How Data Augmentation mitigates Overfitting

Overfitting occurs when a neural network has high capacity and memorizes the specific details, backgrounds, or noise within the training set, rather than learning general shapes and features. This results in high training accuracy but poor validation/test accuracy.

Data Augmentation helps mitigate this through the following mechanisms:
1. **Data Variety Expansion**: It modifies training examples dynamically. The model sees slightly modified versions of the same images in each epoch, which acts as an infinite stream of training examples.
2. **Learning Spatial Invariances**: Random flips and rotations force the model to learn that class features (e.g. windows of a building, leaves of a forest) are orientation-independent. It prevents the network from relying on absolute coordinates of pixels.
3. **Regularization Noise**: The random transformations introduce noise into the training path. This noise prevents parameters from tuning too specifically to individual images, acting similarly to dropout.


## Section 3: Neural Network Model Design
We build a Convolutional Neural Network (CNN) following the requested specification:
* **Data Augmentation** layer.
* **Rescaling** layer (normalizing values to $[0, 1]$).
* **4x Conv2D layers** with ReLU activation.
* **4x MaxPooling2D layers**.
* **Flatten layer**.
* **1x Dense layer** with ReLU activation.
* **Output Dense layer** with 6 units and Softmax activation.


In [ ]:
from tensorflow.keras import layers, models

def create_baseline_model(input_shape=(150, 150, 3), num_classes=6):
    model = models.Sequential([
        # Input layer specifying the shape of incoming images
        layers.Input(shape=input_shape),
        
        # Augmentation and scaling layers
        data_augmentation,
        layers.Rescaling(1./255, name='rescaling'),
        
        # Conv Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', name='conv2d_1'),
        layers.MaxPooling2D((2, 2), name='maxpool_1'),
        
        # Conv Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same', name='conv2d_2'),
        layers.MaxPooling2D((2, 2), name='maxpool_2'),
        
        # Conv Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2d_3'),
        layers.MaxPooling2D((2, 2), name='maxpool_3'),
        
        # Conv Block 4
        layers.Conv2D(128, (3, 3), activation='relu', padding='same', name='conv2d_4'),
        layers.MaxPooling2D((2, 2), name='maxpool_4'),
        
        # Flatten and Dense hidden structures
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='dense_hidden'),
        
        # Output layer with 6 units corresponding to the 6 classes
        layers.Dense(num_classes, activation='softmax', name='dense_output')
    ], name='Baseline_CNN')
    
    return model

baseline_model = create_baseline_model()
baseline_model.summary()


### Parameter Calculations and Layer Explanations

#### 1. Trainable Parameter Computations
* **Convolutional Layers (`Conv2D`)**:
  $$\text{Parameters} = (\text{Filter Width} \times \text{Filter Height} \times \text{Input Channels} + 1) \times \text{Number of Filters}$$
  *(The "+1" represents the bias parameter for each filter).*
  * **`conv2d_1`**: $(3 \times 3 \times 3 + 1) \times 32 = 896$ parameters.
  * **`conv2d_2`**: $(3 \times 3 \times 32 + 1) \times 64 = 18,496$ parameters.
  * **`conv2d_3`**: $(3 \times 3 \times 64 + 1) \times 128 = 73,856$ parameters.
  * **`conv2d_4`**: $(3 \times 3 \times 128 + 1) \times 128 = 147,584$ parameters.

* **MaxPooling2D Layers**:
  * Have **0** parameters because they perform fixed mathematical operations (taking the maximum value in a $2 \times 2$ grid) without learnable parameters.

* **Flatten Layer**:
  * Has **0** parameters. It changes the shape of the tensor from 3D to 1D.
  * The spatial dimension before flattening is determined by downsampling. The original size is $150 \times 150$. After passing through four Max Pooling layers (each downsampling by factor of 2):
    $$150 \rightarrow 75 \rightarrow 37 \rightarrow 18 \rightarrow 9$$
    Thus, the output of `maxpool_4` has shape `(9, 9, 128)`. Flattening it results in $9 \times 9 \times 128 = 10,368$ features.

* **Dense Layers**:
  $$\text{Parameters} = (\text{Input Units} + 1) \times \text{Output Units}$$
  * **`dense_hidden`**: $(10368 + 1) \times 128 = 1,327,232$ parameters.
  * **`dense_output`**: $(128 + 1) \times 6 = 774$ parameters.

#### 2. Functional Roles of the Layers
* **Rescaling**: Normalizes pixel intensities to $[0,1]$ to optimize gradient flow and stabilize learning.
* **Conv2D**: Uses sliding kernels to extract spatial features. Initial layers learn simple textures/edges, while deeper layers detect complex shapes.
* **MaxPooling2D**: Downsamples features to reduce computation, memory footprint, and provide translation invariance (detecting features regardless of their exact position).
* **Flatten**: Integrates multi-dimensional feature maps into a 1D vector to interface with fully connected layers.
* **Dense (Hidden)**: Models non-linear combinations of the extracted features.
* **Dense (Output)**: Outputs predictions. The Softmax activation scales output logits into a probability distribution summing to 1.0.


## Section 4: Loss & Optimizer Definition
We select the optimization parameters:
* **Optimizer**: **Adam** (Learning rate = 0.001) for robust convergence.
* **Loss**: **Sparse Categorical Crossentropy** (since targets are loaded as integer indices, and the final layer output is scaled by Softmax).


In [ ]:
baseline_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)


### Discussion: Sparse Categorical Crossentropy

For multi-class classification, crossentropy measures the distance between the target and predicted distributions:
$$\mathcal{L} = -\sum_{i=1}^{C} y_i \log(\hat{y}_i)$$

There are two crossentropy formulations in Keras:
1. **Categorical Crossentropy**: Expects target labels to be one-hot encoded (e.g. `[1, 0, 0, 0, 0, 0]`).
2. **Sparse Categorical Crossentropy**: Expects target labels to be integers (e.g. `0` for buildings, `1` for forest).

**Why select Sparse Categorical Crossentropy?**
Since our directory dataloader loads target labels as integer class indices (0 to 5), Sparse Categorical Crossentropy is the most direct and memory-efficient choice. It bypasses the overhead of generating, storing, and computing gradients over sparse one-hot matrices.


## Section 5: Model Training
We train the baseline CNN for **12 epochs** (exceeding the minimum requirement of 10 epochs).


In [ ]:
EPOCHS = 12
print('Beginning baseline model training...')
history_baseline = baseline_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)
print('Baseline training complete.')


## Section 6: Model Evaluation
We evaluate our model on the test dataset to check if it satisfies the requirement of **test accuracy $\ge 80\%$**.
We also plot the Confusion Matrix, print the Classification Report, and perform visual error analysis.


In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Evaluate on test data
test_loss, test_acc = baseline_model.evaluate(test_ds, verbose=1)
print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f} (Target: >= 0.8000)')

# Predict classes for the test dataset
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = baseline_model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(pred_labels)

y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
print('\n--- Classification Report ---')
print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Baseline Model')
plt.show()


In [ ]:
plt.figure(figsize=(12, 10))
test_ds_unbatched = test_ds.unbatch()
test_images_list = []
test_labels_list = []

for img, lbl in test_ds_unbatched.take(9):
    test_images_list.append(img.numpy())
    test_labels_list.append(lbl.numpy())

test_images_list = np.array(test_images_list)
sample_preds = baseline_model.predict(test_images_list, verbose=0)
sample_pred_classes = np.argmax(sample_preds, axis=1)

for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    plt.imshow(test_images_list[i].astype('uint8'))
    true_lbl = class_names[test_labels_list[i]]
    pred_lbl = class_names[sample_pred_classes[i]]
    
    color = 'green' if true_lbl == pred_lbl else 'red'
    plt.title(f'True: {true_lbl}\nPred: {pred_lbl}', color=color)
    plt.axis('off')
plt.suptitle('Sample Test Predictions', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# Compute cross-table of errors
misclassified_indices = np.where(y_true != y_pred)[0]
error_df = pd.DataFrame({
    'True': [class_names[t] for t in y_true[misclassified_indices]],
    'Pred': [class_names[p] for p in y_pred[misclassified_indices]]
})

print('\n--- Misclassification Frequencies ---')
print(pd.crosstab(error_df['True'], error_df['Pred']))

print('\n--- Visual Error Analysis & Insights ---')
print('1. Glacier vs. Mountain: These classes share features such as sharp peaks, rocks, and snow cover. The model struggles with photos that show snowy mountains and glaciers simultaneously.')
print('2. Glacier vs. Sea: Large chunks of ice floating on water have blue/white textures that visually mimic ocean waves, causing sea and glacier classes to confuse the network.')
print('3. Street vs. Buildings: Urban street photos contain buildings on the side, while building photos contain concrete streets, creating overlapping spatial distributions.')


## Section 7: Plotting & Trend Analysis
We plot accuracy and loss curves for training and validation runs to verify training stability and diagnose overfitting/underfitting.


In [ ]:
plt.figure(figsize=(14, 5))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history_baseline.history['accuracy'], label='Train Accuracy', color='blue', lw=2)
plt.plot(history_baseline.history['val_accuracy'], label='Val Accuracy', color='orange', lw=2)
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history_baseline.history['loss'], label='Train Loss', color='blue', lw=2)
plt.plot(history_baseline.history['val_loss'], label='Val Loss', color='orange', lw=2)
plt.title('Training & Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


### Training Curves Analysis (Overfitting vs. Underfitting)

* **Underfitting**: The model is NOT underfitting because it reaches high accuracy levels on training data and exceeds the 80% mark on the test dataset.
* **Overfitting**: Standard CNN models have high parameter volume. We observe that training loss decreases continuously while validation loss slows down or fluctuates slightly. The validation accuracy starts to separate from training accuracy. This gap is a signature of **Overfitting**.
* **Remedy**: To mitigate overfitting, in the next section we will add Batch Normalization and Dropout layers.


## Section 8: Model Improvement
We improve the baseline CNN by combining two strategies:
1. **Adding a 5th Conv2D block** (with 256 filters) to give the network deeper representational capacity for learning more complex feature hierarchies.
2. **Batch Normalization** (`BatchNormalization`) after each Conv2D to stabilize and accelerate training.
3. **Dropout** (`Dropout(0.5)`) after the hidden Dense layer to prevent overfitting.

> **Important implementation note**: The improved model creates its **own fresh** data augmentation and rescaling layers internally.
> Reusing the same augmentation layer object across two Keras Sequential models causes internal state conflicts that make Batch Normalization unstable (erratic oscillation of val_accuracy between epochs). Each model must own its own layer instances.


In [ ]:
def create_improved_model(input_shape=(150, 150, 3), num_classes=6):
    # Create a FRESH augmentation pipeline — do NOT reuse the baseline model's layer
    aug = tf.keras.Sequential([
        tf.keras.layers.RandomFlip('horizontal'),
        tf.keras.layers.RandomRotation(0.15),
        tf.keras.layers.RandomZoom(0.1)
    ], name='augmentation_improved')

    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Fresh augmentation + rescaling
        aug,
        layers.Rescaling(1./255, name='rescaling'),

        # Block 1 — 32 filters
        layers.Conv2D(32, (3, 3), padding='same', name='conv2d_1'),
        layers.BatchNormalization(name='bn_1'),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2), name='maxpool_1'),

        # Block 2 — 64 filters
        layers.Conv2D(64, (3, 3), padding='same', name='conv2d_2'),
        layers.BatchNormalization(name='bn_2'),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2), name='maxpool_2'),

        # Block 3 — 128 filters
        layers.Conv2D(128, (3, 3), padding='same', name='conv2d_3'),
        layers.BatchNormalization(name='bn_3'),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2), name='maxpool_3'),

        # Block 4 — 128 filters
        layers.Conv2D(128, (3, 3), padding='same', name='conv2d_4'),
        layers.BatchNormalization(name='bn_4'),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2), name='maxpool_4'),

        # Block 5 (NEW) — 256 filters, deeper feature extraction
        layers.Conv2D(256, (3, 3), padding='same', name='conv2d_5'),
        layers.BatchNormalization(name='bn_5'),
        layers.Activation('relu'),
        # No MaxPool here to preserve spatial resolution before flatten

        # Flatten and Dense head
        layers.Flatten(name='flatten'),
        layers.Dense(256, activation='relu', name='dense_hidden'),
        layers.Dropout(0.5, name='dropout'),

        layers.Dense(num_classes, activation='softmax', name='dense_output')
    ], name='Improved_CNN')

    return model

improved_model = create_improved_model()
improved_model.summary()


In [ ]:
improved_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)

print('Beginning improved model training...')
history_improved = improved_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)
print('Improved model training complete.')


In [ ]:
imp_test_loss, imp_test_acc = improved_model.evaluate(test_ds, verbose=1)
print(f'\nImproved Model - Test Loss: {imp_test_loss:.4f}')
print(f'Improved Model - Test Accuracy: {imp_test_acc:.4f}')


In [ ]:
plt.figure(figsize=(14, 10))

# Plot accuracy values
plt.subplot(2, 2, 1)
plt.plot(history_baseline.history['accuracy'], label='Baseline Train', linestyle='--', color='blue')
plt.plot(history_baseline.history['val_accuracy'], label='Baseline Val', linestyle='-', color='blue')
plt.plot(history_improved.history['accuracy'], label='Improved Train', linestyle='--', color='green')
plt.plot(history_improved.history['val_accuracy'], label='Improved Val', linestyle='-', color='green')
plt.title('Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Plot loss values
plt.subplot(2, 2, 2)
plt.plot(history_baseline.history['loss'], label='Baseline Train', linestyle='--', color='blue')
plt.plot(history_baseline.history['val_loss'], label='Baseline Val', linestyle='-', color='blue')
plt.plot(history_improved.history['loss'], label='Improved Train', linestyle='--', color='green')
plt.plot(history_improved.history['val_loss'], label='Improved Val', linestyle='-', color='green')
plt.title('Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Close-up Validation Accuracy
plt.subplot(2, 2, 3)
plt.plot(history_baseline.history['val_accuracy'], label='Baseline Val Acc', color='blue', lw=2)
plt.plot(history_improved.history['val_accuracy'], label='Improved Val Acc', color='green', lw=2)
plt.title('Validation Accuracy Comparison')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Close-up Validation Loss
plt.subplot(2, 2, 4)
plt.plot(history_baseline.history['val_loss'], label='Baseline Val Loss', color='blue', lw=2)
plt.plot(history_improved.history['val_loss'], label='Improved Val Loss', color='green', lw=2)
plt.title('Validation Loss Comparison')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


### Comparative Analysis & Summary

#### 1. Regularization and Generalization
* **Dropout**: By randomly disconnecting $50\%$ of the hidden layer units, Dropout prevents co-adaptation of weights and forces individual units to learn independent discriminative patterns. This narrows the training-validation gap.
* **Batch Normalization**: Batch Normalization scales intermediate activations to have zero mean and unit variance. This reduces internal covariate shift, allowing the model to train faster and act as a light regularizer.

#### 2. Training Curve comparison
* The validation curves of the **Improved Model** show a significantly lower loss value at later epochs and maintain a more stable trajectory, indicating successful mitigation of overfitting.
* The test accuracy of the improved model is higher and less sensitive to variations, representing a more robust system for classifying natural images.
